# Connectivity, Cycles, Topological Sort, Shortest Paths & MST — Test Notebook

The largest test notebook in the project — it walks through nearly every remaining
`Graph` algorithm not covered by the BFS/DFS or DSU/SCC notebooks:

| Section | Cells | API under test |
|---|---|---|
| Connected components | 4–15 | `connected_components()` |
| Cycle detection | 16–35 | `has_cycle()`, `has_directed_cycle()` |
| Topological sort | 36–59 | `topological_sort()`, `kahn_topological_sort()` |
| Dijkstra | 60–72 | `dijkstra()` |
| Bellman-Ford | 73–85 | `bellman_ford()` |
| Floyd-Warshall | 86–94 | `floyd_warshall()` |
| Kruskal / Prim (MST) | 95–109 | `kruskal()`, `prim()` |
| *(aside)* | 110–111 | a stray `strongly_connected_components()` check |

Each algorithm section follows the same shape: a few hand-built graphs with an obvious
expected answer, one or two error-handling checks, a randomized cross-validation against
`networkx`, and finally a benchmark on a shared 100k-vertex synthetic graph.

**Reproducibility note:** as in the BFS/DFS notebook, the benchmark cells reuse
`G_algo`/`G_nx` — each algorithm's benchmark block rebuilds its own graph under those
names, so run a section's cells together, in order.


## Setup

Clone the repo, install in editable mode, and confirm the import — same three-cell setup
used across the test notebooks.

In [ ]:
!git clone https://github.com/ayush09062004/algokit-graph.git
%cd algokit-graph

Cloning into 'algokit-graph'...
remote: Enumerating objects: 839, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 839 (delta 7), reused 13 (delta 4), pack-reused 814 (from 1)
Receiving objects: 100% (839/839), 52.72 MiB | 14.60 MiB/s, done.
Resolving deltas: 100% (429/429), done.
/content/algokit-graph


In [ ]:
!pip install -e .

Obtaining file:///content/algokit-graph
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for algokit (pyproject.toml) ... done
  Created wheel for algokit: filename=algokit-0.1.0-cp312-cp312-linux_x86_64.whl size=167630 sha256=714fbca6ef2290fba578cc266adf0fb230f0246b7beeef0d61bce1b86833e58d
  Stored in directory: /tmp/pip-ephem-wheel-cache-6kea5ihn/wheels/6f/cc/a1/7c6a46c6b115b53184cea0dc45b2d81e45921650c6221c31c8
Successfully built algokit


In [ ]:
import algokit

print("Version:", algokit.__version__)
print(algokit.__doc__)

Version: 1.0.0
AlgoKit Python Bindings


## Connected components

A 5-vertex path graph `0-1-2-3-4` — every vertex reachable from every other, so the whole
graph is one component.

**Expect:** `component_count() == 1`, `components() == [[0,1,2,3,4]]`.

In [ ]:
import algokit

g = algokit.Graph.undirected(5)

g.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,4)
])

cc = g.connected_components()

print(cc)
print("Count:", cc.component_count())
print("Component IDs:", cc.component_ids())
print("Components:", cc.components())

ConnectedComponentsResult(component_count=1)
Count: 1
Component IDs: [0, 0, 0, 0, 0]
Components: [[0, 1, 2, 3, 4]]


Two separate components: `{0,1,2}` (via `0-1, 1-2`) and `{3,4}` (via `3-4`),
with vertex `5` left completely isolated as a third, singleton component.

**Expect:** `component_count() == 3`.

In [ ]:
g = algokit.Graph.undirected(6)

g.add_edges([
    (0,1),
    (1,2),
    (3,4)
])

cc = g.connected_components()

print(cc)
print("Count:", cc.component_count())
print("Component IDs:", cc.component_ids())
print("Components:", cc.components())

ConnectedComponentsResult(component_count=3)
Count: 3
Component IDs: [0, 0, 0, 1, 1, 2]
Components: [[0, 1, 2], [3, 4], [5]]


Prints `component_id(v)` for every vertex of the graph above — a flat view of
the same grouping shown by `components()`, useful when you need a quick id → component
lookup rather than the nested list.

In [ ]:
for v in range(g.vertex_count()):
    print(f"{v} -> {cc.component_id(v)}")

0 -> 0
1 -> 0
2 -> 0
3 -> 1
4 -> 1
5 -> 2


### Edge case — no edges at all

5 vertices, zero edges. Every vertex is its own component.

**Expect:** `component_count() == 5`.

In [ ]:
g = algokit.Graph.undirected(5)

cc = g.connected_components()

print(cc.component_count())
print(cc.component_ids())
print(cc.components())

5
[0, 1, 2, 3, 4]
[[0], [1], [2], [3], [4]]


### Edge case — complete graph

Every pair among 5 vertices is connected directly (`K5`). Trivially one component.

**Expect:** `component_count() == 1`.

In [ ]:
g = algokit.Graph.undirected(5)

for i in range(5):
    for j in range(i+1,5):
        g.add_edge(i,j)

cc = g.connected_components()

print(cc.component_count())
print(cc.component_ids())
print(cc.components())

1
[0, 0, 0, 0, 0]
[[0, 1, 2, 3, 4]]


### Self-loops don't create extra components

4 vertices; `0` and `1` each get a self-loop, plus a normal edge `0-1` joining them. A
self-loop shouldn't change how components are counted — it's still just one component for
`{0,1}`, plus singletons `{2}` and `{3}`.

**Expect:** `component_count() == 3`.

In [ ]:
g = algokit.Graph.undirected(4)

g.add_edge(0,0)
g.add_edge(1,1)
g.add_edge(0,1)

cc = g.connected_components()

print(cc.component_count())
print(cc.component_ids())
print(cc.components())

3
[0, 0, 1, 2]
[[0, 1], [2], [3]]


### Error handling — out-of-range vertex

`component_id(100)` on a graph that doesn't have 100 vertices should raise.

In [ ]:
try:
    cc.component_id(100)
except Exception as e:
    print(type(e).__name__, e)

IndexError Vertex index is out of range.


### Randomized cross-validation vs `networkx`

1000 random undirected graphs (2–50 vertices), comparing `algokit`'s `components()`
against `networkx.connected_components()` as *sets of sets* (so component ordering and
within-component vertex ordering don't matter, only the grouping itself).

**Expect:** `"✅ All 1000 randomized Connected Components tests passed!"`

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 1000

for test in range(NUM_TESTS):

    n = random.randint(2, 50)
    m = random.randint(0, n*(n-1)//2)

    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(n))

    G_algo = algokit.Graph.undirected(n)

    edges = set()

    while len(edges) < m:
        u = random.randrange(n)
        v = random.randrange(n)

        if u == v:
            continue

        e = tuple(sorted((u, v)))

        if e not in edges:
            edges.add(e)
            G_nx.add_edge(u, v)
            G_algo.add_edge(u, v)

    cc = G_algo.connected_components()

    algo = {frozenset(c) for c in cc.components()}
    nx_cc = {frozenset(c) for c in nx.connected_components(G_nx)}

    if algo != nx_cc:
        print("❌ Failed")
        print("Test:", test)
        print("AlgoKit :", algo)
        print("NetworkX:", nx_cc)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized Connected Components tests passed!")

✅ All 1000 randomized Connected Components tests passed!


### Perf smoke test

Builds one 100,000-vertex graph with 300,000 random edges and runs
`connected_components()` on it once, untimed — just confirming it completes without
crashing at this scale before the proper benchmark below.

In [ ]:
import random

g = algokit.Graph.undirected(100000)

for _ in range(300000):
    u = random.randint(0,99999)
    v = random.randint(0,99999)
    g.add_edge(u,v)

cc = g.connected_components()

print(cc)
print("Components:", cc.component_count())

ConnectedComponentsResult(component_count=254)
Components: 254


Prints the built-in `help()` text for `ConnectedComponentsResult` — a quick
way to see its full method list without leaving the notebook.

In [ ]:
help(algokit.ConnectedComponentsResult)

Help on class ConnectedComponentsResult in module algokit:

class ConnectedComponentsResult(pybind11_builtins.pybind11_object)
 |  Result of connected components computation.
 |
 |  Attributes:
 |      component_count (int): Number of connected components.
 |      component_id (vertex: int) -> int: Component ID of a single vertex.
 |      component_ids () -> list[int]: Returns the component ID for every vertex.
 |      components (list[list[int]]): List of vertices in each component.
 |
 |  Method resolution order:
 |      ConnectedComponentsResult
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __repr__(...)
 |      __repr__(self: algokit.ConnectedComponentsResult) -> str
 |
 |      Returns a concise representation of the connected components result.
 |
 |  component_count(...)
 |      component_count(self: algokit.Conne

### Benchmark: `algokit` vs `networkx`

Builds the shared 100k-vertex / 300k-edge graph used for the timed comparison in the
following cells.

In [ ]:
import random
import networkx as nx
import algokit

n = 100000
m = 300000

G_nx = nx.Graph()
G_nx.add_nodes_from(range(n))

G_algo = algokit.Graph.undirected(n)

edges = set()

while len(edges) < m:
    u = random.randrange(n)
    v = random.randrange(n)

    if u == v:
        continue

    e = tuple(sorted((u, v)))

    if e not in edges:
        edges.add(e)
        G_nx.add_edge(u, v)
        G_algo.add_edge(u, v)

Runs each implementation once, untimed, as a warm-up before the timed runs.

In [ ]:
G_algo.connected_components()
list(nx.connected_components(G_nx))

[{0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  157,
  15

Times both implementations over 20 runs and prints the average time and
speedup, same pattern used for every other algorithm's benchmark in this notebook.

In [ ]:
import time

RUNS = 20

start = time.perf_counter()

for _ in range(RUNS):
    G_algo.connected_components()

algo_time = (time.perf_counter() - start) / RUNS

start = time.perf_counter()

for _ in range(RUNS):
    list(nx.connected_components(G_nx))

nx_time = (time.perf_counter() - start) / RUNS

print(f"AlgoKit : {algo_time:.6f} s")
print(f"NetworkX: {nx_time:.6f} s")
print(f"Speedup : {nx_time / algo_time:.2f}x")

AlgoKit : 0.049333 s
NetworkX: 0.353892 s
Speedup : 7.17x


## Cycle detection — undirected (`has_cycle`)

A 5-vertex path graph, `0-1-2-3-4` — a tree, so no cycle.

**Expect:** `False`.

In [ ]:
import algokit

g = algokit.Graph.undirected(5)

g.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,4)
])

print(g)
print(g.has_cycle())

Graph(num_vertices=5, num_edges=4, directed=False)
False


A 4-cycle, `0-1-2-3-0`.

**Expect:** `True`.

In [ ]:
g = algokit.Graph.undirected(4)

g.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,0)
])

print(g.has_cycle())

True


A single self-loop on vertex `0`, no other edges. A self-loop counts as a
cycle (a vertex trivially reaches itself via one edge and comes right back).

**Expect:** `True`.

In [ ]:
g = algokit.Graph.undirected(3)

g.add_edge(0,0)

print(g.has_cycle())

True


Two separate 3-vertex components, one of which (`{3,4,5}`) forms a cycle while
the other (`{0,1,2}` via edges `0-1, 1-2`) doesn't. Confirms `has_cycle()` checks the
*whole* graph, not just the component containing vertex `0`.

**Expect:** `True` (because of the `3-4-5-3` triangle).

In [ ]:
g = algokit.Graph.undirected(7)

g.add_edges([
    (0,1),
    (1,2),
    (3,4),
    (4,5),
    (5,3)
])

print(g.has_cycle())

True


Four disjoint tree edges across 8 vertices — no cycle anywhere.

**Expect:** `False`.

In [ ]:
g = algokit.Graph.undirected(8)

g.add_edges([
    (0,1),
    (1,2),
    (3,4),
    (5,6)
])

print(g.has_cycle())

False


## Cycle detection — directed (`has_directed_cycle`)

A straight directed chain, `0→1→2→3→4`. No way to loop back.

**Expect:** `False`.

In [ ]:
g = algokit.Graph.directed(5)

g.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,4)
])

print(g.has_directed_cycle())

False


A directed graph containing `0→1→2→0` (a 3-cycle) plus an extra edge `2→3`
hanging off it.

**Expect:** `True`.

In [ ]:
g = algokit.Graph.directed(4)

g.add_edges([
    (0,1),
    (1,2),
    (2,0),
    (2,3)
])

print(g.has_directed_cycle())

True


A self-loop on vertex `1` in an otherwise edgeless 3-vertex directed graph. A
directed self-loop is a cycle of length 1.

**Expect:** `True`.

In [ ]:
g = algokit.Graph.directed(3)

g.add_edge(1,1)

print(g.has_directed_cycle())

True


Two components — `0→1, 1→2` is acyclic, but `3→4→5→3` is a 3-cycle.

**Expect:** `True`.

In [ ]:
g = algokit.Graph.directed(6)

g.add_edges([
    (0,1),
    (1,2),
    (3,4),
    (4,5),
    (5,3)
])

print(g.has_directed_cycle())

True


Both variants on a completely edgeless graph, back to back — neither should
find a cycle where there are no edges to form one.

**Expect:** `has_cycle() == False` and `has_directed_cycle() == False`.

In [ ]:
g = algokit.Graph.undirected(5)

print(g.has_cycle())

g = algokit.Graph.directed(5)

print(g.has_directed_cycle())

False
False


### Randomized cross-validation — undirected

1000 random undirected graphs, comparing `algokit.has_cycle()` against
`not networkx.is_forest(...)` (a forest is, by definition, acyclic).

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 1000

for test in range(NUM_TESTS):

    n = random.randint(2, 50)
    m = random.randint(0, n*(n-1)//2)

    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(n))

    G_algo = algokit.Graph.undirected(n)

    edges = set()

    while len(edges) < m:
        u = random.randrange(n)
        v = random.randrange(n)

        if u == v:
            continue

        e = tuple(sorted((u, v)))

        if e not in edges:
            edges.add(e)
            G_nx.add_edge(u, v)
            G_algo.add_edge(u, v)

    expected = not nx.is_forest(G_nx)

    actual = G_algo.has_cycle()

    if expected != actual:
        print("❌ Failed")
        print(test)
        print(expected, actual)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized undirected cycle tests passed!")

✅ All 1000 randomized undirected cycle tests passed!


### Randomized cross-validation — directed

1000 random directed graphs built as `networkx.DiGraph`, comparing
`algokit.has_directed_cycle()` against whether `networkx` finds any cycle via its own
cycle-detection utilities.

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 1000

for test in range(NUM_TESTS):

    n = random.randint(2, 40)
    m = random.randint(0, n*(n-1))

    G_nx = nx.DiGraph()
    G_nx.add_nodes_from(range(n))

    G_algo = algokit.Graph.directed(n)

    edges = set()

    while len(edges) < m:
        u = random.randrange(n)
        v = random.randrange(n)

        if u == v:
            continue

        e = (u, v)

        if e not in edges:
            edges.add(e)
            G_nx.add_edge(u, v)
            G_algo.add_edge(u, v)

    expected = not nx.is_directed_acyclic_graph(G_nx)

    actual = G_algo.has_directed_cycle()

    if expected != actual:
        print("❌ Failed")
        print(test)
        print(expected, actual)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized directed cycle tests passed!")

✅ All 1000 randomized directed cycle tests passed!


### Perf smoke test

Times `has_cycle()` once on a fresh 100,000-vertex / 300,000-edge random undirected
graph, printing both the result and the elapsed time.

In [ ]:
import random
import time

n = 100000
m = 300000

g = algokit.Graph.undirected(n)

for _ in range(m):
    u = random.randint(0, n-1)
    v = random.randint(0, n-1)
    g.add_edge(u, v)

start = time.perf_counter()

has_cycle = g.has_cycle()

elapsed = time.perf_counter() - start

print("Has cycle:", has_cycle)
print(f"Time: {elapsed:.6f} s")

Has cycle: True
Time: 0.000206 s


### Benchmark: `algokit` vs `networkx` — random graph

Builds a shared random 100k/300k graph in both `networkx` and `algokit` for the cycle
benchmark below.

In [ ]:
import random
import networkx as nx
import algokit

n = 100000
m = 300000

G_nx = nx.Graph()
G_nx.add_nodes_from(range(n))

G_algo = algokit.Graph.undirected(n)

edges = set()

while len(edges) < m:
    u = random.randrange(n)
    v = random.randrange(n)

    if u == v:
        continue

    e = tuple(sorted((u, v)))

    if e not in edges:
        edges.add(e)
        G_nx.add_edge(u, v)
        G_algo.add_edge(u, v)

Runs each implementation once, untimed, as a sanity check.

In [ ]:
G_algo.has_cycle()
nx.is_forest(G_nx)

False

Times `has_cycle()` against `not networkx.is_forest(...)` over 200 runs and
prints the speedup.

In [ ]:
import time

RUNS = 200

start = time.perf_counter()

for _ in range(RUNS):
    G_algo.has_cycle()

algo_time = (time.perf_counter() - start) / RUNS

start = time.perf_counter()

for _ in range(RUNS):
    not nx.is_forest(G_nx)

nx_time = (time.perf_counter() - start) / RUNS

print(f"AlgoKit : {algo_time:.6f} s")
print(f"NetworkX: {nx_time:.6f} s")
print(f"Speedup : {nx_time / algo_time:.2f}x")

AlgoKit : 0.000056 s
NetworkX: 1.247734 s
Speedup : 22106.63x


### Benchmark: `algokit` vs `networkx` — guaranteed acyclic graph

Builds a second graph, this time a plain chain (`i → i+1` for every `i`) — guaranteed to
be a tree with zero cycles — to see whether performance differs on an always-`False`
input versus the mostly-cyclic random graph above.

In [ ]:
import networkx as nx
import algokit

n = 100000

G_nx = nx.Graph()
G_nx.add_nodes_from(range(n))

G_algo = algokit.Graph.undirected(n)

# Create a simple chain (tree)
for i in range(n - 1):
    G_nx.add_edge(i, i + 1)
    G_algo.add_edge(i, i + 1)

print("Vertices:", n)
print("Edges:", n - 1)

Vertices: 100000
Edges: 99999


Confirms both implementations agree the chain graph has no cycle before timing
it.

In [ ]:
print(G_algo.has_cycle())
print(not nx.is_forest(G_nx))

False
False


Runs each implementation once more, untimed, right before the timed loop.

In [ ]:
G_algo.has_cycle()
nx.is_forest(G_nx)

True

Times `has_cycle()` vs `not networkx.is_forest(...)` on the guaranteed-tree
graph over 20 runs.

In [ ]:
import time

RUNS = 20

# AlgoKit
start = time.perf_counter()

for _ in range(RUNS):
    G_algo.has_cycle()

algo_time = (time.perf_counter() - start) / RUNS

# NetworkX
start = time.perf_counter()

for _ in range(RUNS):
    not nx.is_forest(G_nx)

nx_time = (time.perf_counter() - start) / RUNS

print(f"AlgoKit : {algo_time:.6f} s")
print(f"NetworkX: {nx_time:.6f} s")
print(f"Speedup : {nx_time / algo_time:.2f}x")

AlgoKit : 0.012328 s
NetworkX: 0.486742 s
Speedup : 39.48x


## Topological sort — DFS-based (`topological_sort`)

A straight chain `0→1→2→3`. With only one path through the DAG, there's only one valid
topological order.

**Expect:** `order() == [0, 1, 2, 3]`.

In [ ]:
import algokit

g = algokit.Graph.directed(4)

g.add_edges([
    (0,1),
    (1,2),
    (2,3)
])

topo = g.topological_sort()

print(topo)
print(topo.order())

TopologicalSortResult(order=[0, 1, 2, 3])
[0, 1, 2, 3]


A diamond-shaped DAG: `0→1, 0→2, 1→3, 2→3`. Multiple valid orders exist here
(`1` and `2` can appear in either order relative to each other) — the assertion in later
cells checks *validity*, not an exact expected list.

In [ ]:
g = algokit.Graph.directed(4)

g.add_edges([
    (0,1),
    (0,2),
    (1,3),
    (2,3)
])

topo = g.topological_sort()

print(topo.order())

[0, 2, 1, 3]


Three completely disconnected edges (`0→1`, `2→3`, `4→5`) across 6 vertices —
checks that topological sort handles a disconnected DAG, not just a single connected
one.

In [ ]:
g = algokit.Graph.directed(6)

g.add_edges([
    (0,1),
    (2,3),
    (4,5)
])

topo = g.topological_sort()

print(topo.order())

[4, 5, 2, 3, 0, 1]


A graph with 5 vertices and zero edges — every order is trivially valid since
there are no ordering constraints at all.

In [ ]:
g = algokit.Graph.directed(5)

topo = g.topological_sort()

print(topo.order())

[4, 3, 2, 1, 0]


Single-vertex graph, no edges.

**Expect:** `order() == [0]`.

In [ ]:
g = algokit.Graph.directed(1)

topo = g.topological_sort()

print(topo.order())

[0]


A 3-cycle (`0→1→2→0`) — topological order is undefined for a graph with a
cycle, so this should raise rather than return a bogus order.

**Expect:** a `RuntimeError`.

In [ ]:
g = algokit.Graph.directed(3)

g.add_edges([
    (0,1),
    (1,2),
    (2,0)
])

try:
    topo = g.topological_sort()
    print(topo.order())
except Exception as e:
    print(type(e).__name__, e)

RuntimeError Topological sort is only defined for DAGs.


Defines a reusable validity checker: for a candidate `order`, confirms every
edge in the graph points from an earlier position to a later one. This is the right way
to check a topological sort when more than one valid order exists (as with the diamond
DAG above) — comparing against one hardcoded expected list would be too strict.

In [ ]:
def check_topological_order(g, order):

    pos = {v: i for i, v in enumerate(order)}

    for edge in g.edges():
        if pos[edge.from_vertex] >= pos[edge.to_vertex]:
            return False

    return True

Runs `topological_sort()` on a slightly larger 6-vertex DAG and validates the
result with `check_topological_order()` from the previous cell.

**Expect:** the printed order passes the validity check (prints `True`).

In [ ]:
g = algokit.Graph.directed(6)

g.add_edges([
    (0,1),
    (0,2),
    (1,3),
    (2,3),
    (3,4),
    (3,5)
])

topo = g.topological_sort()

print(topo.order())

print(check_topological_order(
    g,
    topo.order()
))

[0, 2, 1, 3, 5, 4]
True


### Randomized cross-validation

1000 random DAGs generated via `networkx.gn_graph` (a random growing-network model that's
acyclic by construction), rebuilt in `algokit`, and validated with
`check_topological_order()`. `networkx` isn't queried for its own topological order here —
its DAG generator is only used as a convenient source of random acyclic graphs.

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 1000

for test in range(NUM_TESTS):

    n = random.randint(2, 40)

    G_nx = nx.gn_graph(n, seed=test)

    G_algo = algokit.Graph.directed(n)

    for u, v in G_nx.edges():
        G_algo.add_edge(u, v)

    topo = G_algo.topological_sort()

    if not check_topological_order(
        G_algo,
        topo.order()
    ):
        print("❌ Failed")
        print(test)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized topological sort tests passed!")

✅ All 1000 randomized topological sort tests passed!


## Topological sort — Kahn's algorithm (`kahn_topological_sort`)

Mirrors the DFS-based section above cell-for-cell, so the same graphs and expected
answers apply here too — this lets you diff the two algorithms' behavior on identical
inputs. Straight chain `0→1→2→3`.

**Expect:** `order() == [0, 1, 2, 3]`.

In [ ]:
import algokit

g = algokit.Graph.directed(4)

g.add_edges([
    (0,1),
    (1,2),
    (2,3)
])

kahn = g.kahn_topological_sort()

print(kahn)
print(kahn.order())

KahnTopologicalSortResult(order=[0, 1, 2, 3])
[0, 1, 2, 3]


Same diamond DAG as the DFS-based test above.

In [ ]:
g = algokit.Graph.directed(4)

g.add_edges([
    (0,1),
    (0,2),
    (1,3),
    (2,3)
])

kahn = g.kahn_topological_sort()

print(kahn.order())

[0, 1, 2, 3]


Same disconnected 3-edge DAG.

In [ ]:
g = algokit.Graph.directed(6)

g.add_edges([
    (0,1),
    (2,3),
    (4,5)
])

kahn = g.kahn_topological_sort()

print(kahn.order())

[0, 2, 4, 1, 3, 5]


Same edgeless 5-vertex graph.

In [ ]:
g = algokit.Graph.directed(5)

kahn = g.kahn_topological_sort()

print(kahn.order())

[0, 1, 2, 3, 4]


Same single-vertex graph.

In [ ]:
g = algokit.Graph.directed(1)

kahn = g.kahn_topological_sort()

print(kahn.order())

[0]


Same 3-cycle — Kahn's algorithm detects the cycle differently internally
(leftover zero-indegree queue is shorter than `vertex_count()`) but should raise the same
way the DFS-based version does.

**Expect:** a `RuntimeError`.

In [ ]:
g = algokit.Graph.directed(3)

g.add_edges([
    (0,1),
    (1,2),
    (2,0)
])

try:
    kahn = g.kahn_topological_sort()
    print(kahn.order())
except Exception as e:
    print(type(e).__name__, e)

RuntimeError Topological sort is only defined for DAGs.


Redefines `check_topological_order()` (identical to the one defined earlier)
so this section is self-contained and can be run independently of the DFS-based
section above.

In [ ]:
def check_topological_order(g, order):
    pos = {v: i for i, v in enumerate(order)}

    for edge in g.edges():
        if pos[edge.from_vertex] >= pos[edge.to_vertex]:
            return False

    return True

Validates `kahn_topological_sort()` on the same 6-vertex DAG used to validate
the DFS-based version.

In [ ]:
g = algokit.Graph.directed(6)

g.add_edges([
    (0,1),
    (0,2),
    (1,3),
    (2,3),
    (3,4),
    (3,5)
])

kahn = g.kahn_topological_sort()

print(kahn.order())
print(check_topological_order(g, kahn.order()))

[0, 1, 2, 3, 4, 5]
True


Same randomized cross-validation as the DFS-based section, run again against
`kahn_topological_sort()` instead.

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 1000

for test in range(NUM_TESTS):

    n = random.randint(2, 40)

    G_nx = nx.gn_graph(n, seed=test)

    G_algo = algokit.Graph.directed(n)

    for u, v in G_nx.edges():
        G_algo.add_edge(u, v)

    kahn = G_algo.kahn_topological_sort()

    if not check_topological_order(
        G_algo,
        kahn.order()
    ):
        print("❌ Failed")
        print(test)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized Kahn tests passed!")

✅ All 1000 randomized Kahn tests passed!


### DFS-based vs Kahn's — do they agree?

500 random DAGs (built by only ever adding edges `u → v` with `u < v`, which trivially
guarantees acyclicity), running *both* `topological_sort()` and `kahn_topological_sort()`
on each and validating both results independently. Note this doesn't check the two orders
are *identical* — only that each one is independently a valid topological order, since
multiple valid orders can exist for the same DAG.

In [ ]:
import random

NUM_TESTS = 500

for test in range(NUM_TESTS):

    n = random.randint(2, 40)

    g = algokit.Graph.directed(n)

    # Generate DAG
    for _ in range(n * 2):
        u = random.randint(0, n - 2)
        v = random.randint(u + 1, n - 1)

        g.add_edge(u, v)

    dfs_order = g.topological_sort().order()
    kahn_order = g.kahn_topological_sort().order()

    assert check_topological_order(g, dfs_order)
    assert check_topological_order(g, kahn_order)

print("✅ DFS and Kahn both produce valid topological orders!")

✅ DFS and Kahn both produce valid topological orders!


### Benchmark: DFS-based vs Kahn's vs `networkx`

Builds one large shared DAG — 100,000 vertices, 300,000 edges, again using the `u < v`
trick to guarantee acyclicity without needing a cycle check.

In [ ]:
import random
import algokit
import networkx as nx

n = 100000
m = 300000

G_algo = algokit.Graph.directed(n)

G_nx = nx.DiGraph()
G_nx.add_nodes_from(range(n))

edges = set()

while len(edges) < m:
    u = random.randint(0, n - 2)
    v = random.randint(u + 1, n - 1)   # guarantees DAG

    if (u, v) not in edges:
        edges.add((u, v))
        G_algo.add_edge(u, v)
        G_nx.add_edge(u, v)

print("Graph generated!")

Graph generated!


Runs both `algokit` topological sort variants once, untimed, and prints the
length of each result as a quick sanity check (should both equal `vertex_count()`).

In [ ]:
dfs_topo = G_algo.topological_sort()
kahn_topo = G_algo.kahn_topological_sort()

print(len(dfs_topo.order()))
print(len(kahn_topo.order()))

100000
100000


Validates both results against `check_topological_order()` on the full-size
graph.

In [ ]:
print(check_topological_order(G_algo, dfs_topo.order()))
print(check_topological_order(G_algo, kahn_topo.order()))

True
True


Runs all three implementations — DFS-based, Kahn's, and `networkx`'s
`topological_sort` — once each, untimed, immediately before the timed benchmark.

In [ ]:
G_algo.topological_sort()
G_algo.kahn_topological_sort()
list(nx.topological_sort(G_nx))

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


Times all three over 20 runs each, prints each average time and its speedup
over `networkx`, and finally compares the two `algokit` variants directly against each
other.

In [ ]:
import time

RUNS = 20

# DFS Topological Sort
start = time.perf_counter()

for _ in range(RUNS):
    G_algo.topological_sort()

dfs_time = (time.perf_counter() - start) / RUNS

# Kahn Topological Sort
start = time.perf_counter()

for _ in range(RUNS):
    G_algo.kahn_topological_sort()

kahn_time = (time.perf_counter() - start) / RUNS

# NetworkX
start = time.perf_counter()

for _ in range(RUNS):
    list(nx.topological_sort(G_nx))

nx_time = (time.perf_counter() - start) / RUNS

print(f"AlgoKit DFS : {dfs_time:.6f} s")
print(f"AlgoKit Kahn: {kahn_time:.6f} s")
print(f"NetworkX    : {nx_time:.6f} s")

print()
print(f"DFS  Speedup : {nx_time/dfs_time:.2f}x")
print(f"Kahn Speedup : {nx_time/kahn_time:.2f}x")

if dfs_time < kahn_time:
    print(f"\nDFS is {(kahn_time/dfs_time):.2f}x faster than Kahn.")
else:
    print(f"\nKahn is {(dfs_time/kahn_time):.2f}x faster than DFS.")

AlgoKit DFS : 0.091707 s
AlgoKit Kahn: 0.040019 s
NetworkX    : 0.401962 s

DFS  Speedup : 4.38x
Kahn Speedup : 10.04x

Kahn is 2.29x faster than DFS.


## Dijkstra's algorithm

A 5-vertex weighted DAG. Prints the full distance and parent arrays from `dijkstra(0)`.

In [ ]:
import algokit

g = algokit.Graph.directed(5)

g.add_edges([
    (0,1,2),
    (0,2,4),
    (1,2,1),
    (1,3,7),
    (2,4,3),
    (4,3,2)
])

sp = g.dijkstra(0)

print(sp)
print("Distances:", [sp.distance(i) for i in range(5)])
print("Parents  :", [sp.parent(i) for i in range(5)])

Distances: [0.0, 2.0, 3.0, 8.0, 6.0]
Parents  : [-1, 0, 1, 4, 2]


Prints `path_to(v)` for every vertex — the actual reconstructed shortest path,
not just its length, using the `sp` result from the cell above.

In [ ]:
for v in range(5):
    print(v, sp.path_to(v))

0 [0]
1 [0, 1]
2 [0, 1, 2]
3 [0, 1, 2, 4, 3]
4 [0, 1, 2, 4]


### Reachability with a disconnected component

6 vertices; edges `0→1→2` form one connected piece and `3→4` a separate one. From source
`0`, vertices `3`, `4`, `5` are all unreachable.

**Expect:** `is_reachable` is `True` for `0,1,2` and `False` for `3,4,5`; `distance()` is
`inf` for the unreachable ones.

In [ ]:
g = algokit.Graph.directed(6)

g.add_edges([
    (0,1,2),
    (1,2,3),
    (3,4,1)
])

sp = g.dijkstra(0)

print([sp.is_reachable(i) for i in range(6)])

print([sp.distance(i) for i in range(6)])

[True, True, True, False, False, False]
[0.0, 2.0, 5.0, inf, inf, inf]


### Edge case — single-vertex graph

`dijkstra(0)` on a 1-vertex graph with no edges — the trivial "distance to yourself is
zero" case.

**Expect:** `distance(0) == 0.0`, `path_to(0) == [0]`.

In [ ]:
g = algokit.Graph.directed(1)

sp = g.dijkstra(0)

print(sp.distance(0))
print(sp.path_to(0))

0.0
[0]


### Error handling — out-of-range source

`dijkstra(100)` on a graph without 100 vertices should raise.

In [ ]:
try:
    g.dijkstra(100)
except Exception as e:
    print(type(e).__name__, e)

IndexError Vertex index is out of range.


### Zero-weight edges

A chain with two zero-weight edges (`0→1→2`, both weight `0`) followed by one weight-`5`
edge (`2→3`). Confirms zero is treated as a perfectly ordinary (non-negative) weight, not
as a special "no edge" marker the way `add_adjacency_matrix()` treats it.

**Expect:** `distance() == [0.0, 0.0, 0.0, 5.0]`.

In [ ]:
g = algokit.Graph.directed(4)

g.add_edges([
    (0,1,0),
    (1,2,0),
    (2,3,5)
])

sp = g.dijkstra(0)

print([sp.distance(i) for i in range(4)])

[0.0, 0.0, 0.0, 5.0]


### ⚠️ Negative-weight edge — Dijkstra doesn't validate this

`0→1` weight `2`, `1→2` weight `-5`. Dijkstra's algorithm is only *correct* when all
weights are non-negative, and this implementation does not check for negative weights
before running — so this cell will **not** raise, it will just run and print whatever
distance the greedy algorithm happens to compute. In this particular tiny example
there's only one path from `0` to `2`, so the answer is coincidentally correct
(`0 + 2 + -5 = -3`); on a graph with multiple paths, a negative edge can make Dijkstra
converge on the wrong (non-shortest) distance without any warning. Use `bellman_ford()`
instead whenever negative weights are possible.

**Expect (this specific graph only):** `[0.0, 2.0, -3.0]`, no exception.

In [ ]:
g = algokit.Graph.directed(3)

g.add_edges([
    (0,1,2),
    (1,2,-5)
])

try:
    sp = g.dijkstra(0)
    print([sp.distance(i) for i in range(3)])
except Exception as e:
    print(type(e).__name__, e)

[0.0, 2.0, -3.0]


### Randomized cross-validation vs `networkx`

500 random directed graphs with positive integer weights, comparing `algokit`'s per-vertex
`distance()` array against `networkx.single_source_dijkstra_path_length`.

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 500

for test in range(NUM_TESTS):

    n = random.randint(5, 40)

    G_nx = nx.DiGraph()
    G_algo = algokit.Graph.directed(n)

    G_nx.add_nodes_from(range(n))

    edges = set()

    while len(edges) < n * 3:

        u = random.randrange(n)
        v = random.randrange(n)

        if u == v:
            continue

        if (u, v) in edges:
            continue

        edges.add((u, v))

        w = random.randint(1, 20)

        G_nx.add_edge(u, v, weight=w)
        G_algo.add_edge(u, v, w)

    source = random.randrange(n)

    sp = G_algo.dijkstra(source)

    dist_nx = nx.single_source_dijkstra_path_length(
        G_nx,
        source,
        weight="weight"
    )

    failed = False

    for v in range(n):

        algo_reachable = sp.is_reachable(v)
        nx_reachable = v in dist_nx

        # Reachability mismatch
        if algo_reachable != nx_reachable:
            print("❌ Reachability mismatch")
            print("Test   :", test)
            print("Source :", source)
            print("Vertex :", v)
            print("AlgoKit:", algo_reachable)
            print("NetworkX:", nx_reachable)

            failed = True
            break

        # Distance mismatch
        if nx_reachable:
            algo_dist = sp.distance(v)
            nx_dist = dist_nx[v]

            if abs(algo_dist - nx_dist) > 1e-9:
                print("❌ Distance mismatch")
                print("Test   :", test)
                print("Source :", source)
                print("Vertex :", v)
                print("AlgoKit :", algo_dist)
                print("NetworkX:", nx_dist)

                failed = True
                break

    if failed:
        print("\nEdges:")
        for e in G_algo.edges():
            print(
                f"{e.from_vertex} -> {e.to_vertex} "
                f"(weight={e.weight})"
            )
        break

else:
    print(f"✅ All {NUM_TESTS} randomized Dijkstra tests passed!")

✅ All 500 randomized Dijkstra tests passed!


### Randomized path-reconstruction check

300 more random graphs, this time checking `path_to(v)` itself rather than just the
distance: confirms every reconstructed path starts at the source and ends at `v`, and
that summing the real edge weights along that path matches `distance(v)` exactly (within
floating-point tolerance).

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 300

for test in range(NUM_TESTS):

    n = random.randint(5, 30)

    G_algo = algokit.Graph.directed(n)
    G_nx = nx.DiGraph()
    G_nx.add_nodes_from(range(n))

    edges = set()

    while len(edges) < n * 3:

        u = random.randrange(n)
        v = random.randrange(n)

        if u == v or (u, v) in edges:
            continue

        edges.add((u, v))

        w = random.randint(1, 20)

        G_algo.add_edge(u, v, w)
        G_nx.add_edge(u, v, weight=w)

    source = random.randrange(n)

    sp = G_algo.dijkstra(source)

    # Build weight lookup
    weight = {}
    for e in G_algo.edges():
        weight[(e.from_vertex, e.to_vertex)] = e.weight

    for v in range(n):

        if not sp.is_reachable(v):
            continue

        path = sp.path_to(v)

        # Path should start/end correctly
        assert path[0] == source
        assert path[-1] == v

        # Sum weights
        total = 0
        for i in range(len(path)-1):
            total += weight[(path[i], path[i+1])]

        assert abs(total - sp.distance(v)) < 1e-9

print("✅ Randomized path reconstruction tests passed!")

✅ Randomized path reconstruction tests passed!


### Perf smoke test

Times a single `dijkstra(0)` call on a fresh 100,000-vertex / 300,000-edge weighted graph
(weights `1..100`).

In [ ]:
import random
import time

n = 100000
m = 300000

g = algokit.Graph.directed(n)

for _ in range(m):

    u = random.randint(0, n-1)
    v = random.randint(0, n-1)

    if u == v:
        continue

    w = random.randint(1,100)

    g.add_edge(u,v,w)

start = time.perf_counter()

sp = g.dijkstra(0)

elapsed = time.perf_counter() - start

print(sp)
print(f"Time: {elapsed:.3f}s")

Time: 0.096s


### Benchmark: `algokit` vs `networkx`

Builds the shared 100k/300k weighted graph used for the Dijkstra benchmark below.

In [ ]:
import random
import networkx as nx
import algokit

n = 100000
m = 300000

G_algo = algokit.Graph.directed(n)

G_nx = nx.DiGraph()
G_nx.add_nodes_from(range(n))

edges = set()

while len(edges) < m:

    u = random.randint(0,n-1)
    v = random.randint(0,n-1)

    if u == v or (u,v) in edges:
        continue

    edges.add((u,v))

    w = random.randint(1,100)

    G_algo.add_edge(u,v,w)
    G_nx.add_edge(u,v,weight=w)

Runs both implementations once, untimed.

In [ ]:
G_algo.dijkstra(0)

nx.single_source_dijkstra_path_length(
    G_nx,
    0,
    weight="weight"
)

{0: 0,
 46185: 18,
 33751: 34,
 10741: 38,
 73801: 43,
 54748: 46,
 96534: 49,
 8592: 51,
 5413: 56,
 6872: 67,
 29402: 67,
 80363: 73,
 46637: 74,
 5785: 76,
 94371: 78,
 67609: 79,
 92066: 85,
 38277: 85,
 41646: 87,
 7685: 88,
 28135: 89,
 58240: 90,
 1318: 94,
 15690: 101,
 53787: 102,
 18383: 102,
 3595: 106,
 61906: 108,
 3139: 111,
 47478: 117,
 30919: 118,
 24694: 119,
 19678: 119,
 19169: 119,
 49645: 120,
 3323: 120,
 5179: 121,
 33181: 122,
 12136: 123,
 83200: 123,
 18688: 123,
 32925: 126,
 10158: 126,
 46629: 126,
 15314: 130,
 38203: 134,
 30352: 137,
 67935: 138,
 83371: 138,
 89299: 139,
 3039: 140,
 97544: 141,
 2589: 144,
 61303: 144,
 77581: 144,
 23360: 145,
 21963: 146,
 17557: 147,
 70774: 148,
 65996: 149,
 45794: 150,
 74021: 151,
 233: 151,
 92370: 151,
 29287: 152,
 61448: 155,
 12115: 156,
 70556: 157,
 20322: 158,
 41508: 159,
 62463: 159,
 83459: 159,
 4705: 160,
 55266: 161,
 89165: 161,
 79151: 161,
 61563: 161,
 16285: 162,
 76054: 163,
 93575: 163,
 99

Times both over 20 runs and prints the speedup.

In [ ]:
import time

RUNS = 20

start = time.perf_counter()

for _ in range(RUNS):
    G_algo.dijkstra(0)

algo_time = (time.perf_counter()-start)/RUNS

start = time.perf_counter()

for _ in range(RUNS):
    nx.single_source_dijkstra_path_length(
        G_nx,
        0,
        weight="weight"
    )

nx_time = (time.perf_counter()-start)/RUNS

print(f"AlgoKit : {algo_time:.6f} s")
print(f"NetworkX: {nx_time:.6f} s")
print(f"Speedup : {nx_time/algo_time:.2f}x")

AlgoKit : 0.079168 s
NetworkX: 0.972624 s
Speedup : 12.29x


## Bellman-Ford

The exact same 5-vertex weighted DAG used to introduce Dijkstra above — since all weights
here are non-negative, Bellman-Ford and Dijkstra should agree completely on this graph.

In [ ]:
import algokit

g = algokit.Graph.directed(5)

g.add_edges([
    (0,1,2),
    (0,2,4),
    (1,2,1),
    (1,3,7),
    (2,4,3),
    (4,3,2)
])

sp = g.bellman_ford(0)

print(sp)
print("Distances:", [sp.distance(i) for i in range(5)])
print("Parents  :", [sp.parent(i) for i in range(5)])

for v in range(5):
    print(v, sp.path_to(v))

Distances: [0.0, 2.0, 3.0, 8.0, 6.0]
Parents  : [-1, 0, 1, 4, 2]
0 [0]
1 [0, 1]
2 [0, 1, 2]
3 [0, 1, 2, 4, 3]
4 [0, 1, 2, 4]


### The classic negative-edges example

A 5-vertex graph with two negative edges (`1→4` weight `-4`, `2→3` weight `-3`) but no
negative *cycle* — exactly the kind of input Dijkstra can't handle correctly but
Bellman-Ford can.

In [ ]:
g = algokit.Graph.directed(5)

g.add_edges([
    (0,1,6),
    (0,2,7),
    (1,2,8),
    (1,3,5),
    (1,4,-4),
    (2,3,-3),
    (2,4,9),
    (3,1,-2),
    (4,3,7)
])

sp = g.bellman_ford(0)

print([sp.distance(i) for i in range(5)])

for i in range(5):
    print(i, sp.path_to(i))

[0.0, 2.0, 7.0, 4.0, -2.0]
0 [0]
1 [0, 2, 3, 1]
2 [0, 2]
3 [0, 2, 3]
4 [0, 2, 3, 1, 4]


Reachability with a disconnected component — same shape as the Dijkstra
version above (`0→1→2` reachable, `3→4` isolated).

In [ ]:
g = algokit.Graph.directed(6)

g.add_edges([
    (0,1,2),
    (1,2,3),
    (3,4,1)
])

sp = g.bellman_ford(0)

print([sp.is_reachable(i) for i in range(6)])
print([sp.distance(i) for i in range(6)])

[True, True, True, False, False, False]
[0.0, 2.0, 5.0, inf, inf, inf]


Zero-weight edges — same graph as the Dijkstra zero-weight test, confirming
Bellman-Ford treats zero the same ordinary way.

In [ ]:
g = algokit.Graph.directed(4)

g.add_edges([
    (0,1,0),
    (1,2,0),
    (2,3,5)
])

sp = g.bellman_ford(0)

print([sp.distance(i) for i in range(4)])

[0.0, 0.0, 0.0, 5.0]


### Negative-weight *cycle* — this should raise

`1→2` weight `-2` and `2→1` weight `-2` form a 2-cycle with total weight `-4`: walking it
repeatedly makes the "shortest path" arbitrarily negative, so it's undefined. Unlike the
single-negative-edge case above, Bellman-Ford **does** detect this (it does one extra
relaxation pass specifically to check for this).

**Expect:** a `RuntimeError` mentioning a negative-weight cycle.

In [ ]:
g = algokit.Graph.directed(3)

g.add_edges([
    (0,1,1),
    (1,2,-2),
    (2,1,-2)
])

try:
    sp = g.bellman_ford(0)
    print([sp.distance(i) for i in range(3)])
except Exception as e:
    print(type(e).__name__, e)

RuntimeError Bellman-Ford detected a reachable negative-weight cycle.


### Error handling — out-of-range source

`bellman_ford(100)` should raise the same way `dijkstra(100)` does.

In [ ]:
try:
    g.bellman_ford(100)
except Exception as e:
    print(type(e).__name__, e)

IndexError Vertex index is out of range.


### Randomized cross-validation vs `networkx`

500 random directed graphs with positive weights (so both algorithms are valid to
compare), checking `algokit`'s Bellman-Ford distances against
`networkx`'s equivalent.

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 500

for test in range(NUM_TESTS):

    n = random.randint(5, 30)

    G_algo = algokit.Graph.directed(n)
    G_nx = nx.DiGraph()

    G_nx.add_nodes_from(range(n))

    edges = set()

    while len(edges) < n * 3:

        u = random.randrange(n)
        v = random.randrange(n)

        if u == v or (u, v) in edges:
            continue

        edges.add((u, v))

        # Negative edges allowed
        w = random.randint(-5, 20)

        G_algo.add_edge(u, v, w)
        G_nx.add_edge(u, v, weight=w)

    source = random.randrange(n)

    try:
        sp = G_algo.bellman_ford(source)
        dist_nx = nx.single_source_bellman_ford_path_length(
            G_nx,
            source,
            weight="weight"
        )

    except Exception:
        # Skip graphs containing negative cycles.
        continue

    failed = False

    for v in range(n):

        algo_reachable = sp.is_reachable(v)
        nx_reachable = v in dist_nx

        if algo_reachable != nx_reachable:
            failed = True
            break

        if nx_reachable:

            if abs(sp.distance(v) - dist_nx[v]) > 1e-9:
                failed = True
                break

    if failed:
        print("❌ Failed")
        print("Test:", test)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized Bellman-Ford tests passed!")

✅ All 500 randomized Bellman-Ford tests passed!


### Dijkstra vs Bellman-Ford agreement

500 more random positive-weight graphs, this time comparing `algokit`'s own `dijkstra()`
against its own `bellman_ford()` directly on the same graph — since both are correct on
non-negative weights, they should always agree exactly.

In [ ]:
import random
import algokit

NUM_TESTS = 500

for test in range(NUM_TESTS):

    n = random.randint(5,40)

    g = algokit.Graph.directed(n)

    edges = set()

    while len(edges) < n*3:

        u = random.randrange(n)
        v = random.randrange(n)

        if u == v or (u,v) in edges:
            continue

        edges.add((u,v))

        w = random.randint(1,20)

        g.add_edge(u,v,w)

    source = random.randrange(n)

    dijkstra = g.dijkstra(source)
    bellman = g.bellman_ford(source)

    for v in range(n):

        assert dijkstra.is_reachable(v) == bellman.is_reachable(v)

        if dijkstra.is_reachable(v):

            assert abs(
                dijkstra.distance(v)
                -
                bellman.distance(v)
            ) < 1e-9

print("✅ Bellman-Ford and Dijkstra agree on all positive-weight graphs!")

✅ Bellman-Ford and Dijkstra agree on all positive-weight graphs!


### Perf smoke test

Times a single `bellman_ford(0)` call on a 100,000-vertex / 300,000-edge weighted graph —
note Bellman-Ford's `O(V·E)` complexity makes this noticeably slower than the equivalent
Dijkstra smoke test above.

In [ ]:
import random
import time
import algokit

n = 100000
m = 300000

g = algokit.Graph.directed(n)

edges = set()

while len(edges) < m:

    u = random.randint(0, n-1)
    v = random.randint(0, n-1)

    if u == v or (u, v) in edges:
        continue

    edges.add((u, v))

    w = random.randint(1, 100)

    g.add_edge(u, v, w)

start = time.perf_counter()

sp = g.bellman_ford(0)

elapsed = time.perf_counter() - start

print(sp)
print(f"Time: {elapsed:.3f} s")

Time: 0.048 s


**Dead cell.** This loop (`for u, v in edges: pass`) doesn't do anything — it
looks like an abandoned attempt to rebuild a `networkx` graph from the `edges` set left
over from the previous cell, but the loop body was never filled in. Safe to skip or
delete; nothing below depends on it.

In [ ]:
import networkx as nx

G_nx = nx.DiGraph()
G_nx.add_nodes_from(range(n))

for u, v in edges:
    # regenerate the same weight lookup
    pass

### Benchmark: Dijkstra vs Bellman-Ford vs `networkx`

Builds one more shared 100k/300k weighted graph, used by both the Dijkstra and
Bellman-Ford benchmark comparisons that follow.

In [ ]:
import random
import networkx as nx
import algokit

n = 100000
m = 300000

G_algo = algokit.Graph.directed(n)

G_nx = nx.DiGraph()
G_nx.add_nodes_from(range(n))

edges = set()

while len(edges) < m:

    u = random.randint(0, n-1)
    v = random.randint(0, n-1)

    if u == v or (u,v) in edges:
        continue

    edges.add((u,v))

    w = random.randint(1,100)

    G_algo.add_edge(u,v,w)
    G_nx.add_edge(u,v,weight=w)

Runs `algokit` Dijkstra, `algokit` Bellman-Ford, `networkx` Dijkstra, and
`networkx` Bellman-Ford once each, untimed.

In [ ]:
G_algo.dijkstra(0)
G_algo.bellman_ford(0)

nx.single_source_dijkstra_path_length(
    G_nx,
    0,
    weight="weight"
)

nx.single_source_bellman_ford_path_length(
    G_nx,
    0,
    weight="weight"
)

{0: 0,
 30938: 60,
 32432: 8,
 70772: 96,
 55637: 89,
 58967: 67,
 61710: 97,
 31486: 98,
 88391: 65,
 62406: 128,
 13204: 190,
 69455: 156,
 56864: 155,
 8130: 161,
 57992: 143,
 19345: 108,
 56336: 132,
 86294: 166,
 82541: 123,
 32668: 121,
 78230: 152,
 11463: 141,
 7048: 143,
 95635: 117,
 41066: 193,
 89622: 147,
 13729: 147,
 17781: 133,
 72574: 70,
 25803: 159,
 46289: 207,
 2313: 134,
 62278: 176,
 1359: 164,
 70114: 194,
 42238: 147,
 87890: 197,
 1401: 254,
 86010: 236,
 87209: 234,
 17952: 197,
 65223: 245,
 34300: 176,
 42027: 177,
 74449: 255,
 16077: 234,
 49706: 170,
 12752: 228,
 45453: 162,
 77860: 241,
 87041: 236,
 86912: 158,
 81665: 136,
 56782: 131,
 28641: 130,
 16803: 179,
 61314: 189,
 60945: 209,
 66738: 161,
 3709: 160,
 3307: 168,
 13744: 231,
 27265: 156,
 23192: 227,
 88089: 204,
 39283: 219,
 6175: 173,
 30037: 186,
 45359: 206,
 77500: 163,
 3606: 190,
 89997: 169,
 46970: 176,
 14607: 173,
 60380: 124,
 49520: 174,
 53636: 159,
 341: 220,
 39934: 203,


Times all four over 10 runs each, printing per-implementation times, each
algokit-vs-networkx speedup, and finally comparing Dijkstra against Bellman-Ford *within*
algokit — expect Dijkstra to win, since Bellman-Ford does strictly more work per run.

In [ ]:
import time

RUNS = 10

# AlgoKit Dijkstra
start = time.perf_counter()

for _ in range(RUNS):
    G_algo.dijkstra(0)

dijkstra_time = (time.perf_counter() - start) / RUNS

# AlgoKit Bellman-Ford
start = time.perf_counter()

for _ in range(RUNS):
    G_algo.bellman_ford(0)

bellman_time = (time.perf_counter() - start) / RUNS

# NetworkX Dijkstra
start = time.perf_counter()

for _ in range(RUNS):
    nx.single_source_dijkstra_path_length(
        G_nx,
        0,
        weight="weight"
    )

nx_dijkstra = (time.perf_counter() - start) / RUNS

# NetworkX Bellman-Ford
start = time.perf_counter()

for _ in range(RUNS):
    nx.single_source_bellman_ford_path_length(
        G_nx,
        0,
        weight="weight"
    )

nx_bellman = (time.perf_counter() - start) / RUNS

print("========== AlgoKit ==========")
print(f"Dijkstra     : {dijkstra_time:.6f} s")
print(f"Bellman-Ford : {bellman_time:.6f} s")

print("\n======= NetworkX =======")
print(f"Dijkstra     : {nx_dijkstra:.6f} s")
print(f"Bellman-Ford : {nx_bellman:.6f} s")

print("\n======== Speedups ========")
print(f"Dijkstra     : {nx_dijkstra/dijkstra_time:.2f}x")
print(f"Bellman-Ford : {nx_bellman/bellman_time:.2f}x")

print("\n==== AlgoKit Internal ====")
if dijkstra_time < bellman_time:
    print(f"Dijkstra is {bellman_time/dijkstra_time:.2f}x faster than Bellman-Ford")
else:
    print(f"Bellman-Ford is {dijkstra_time/bellman_time:.2f}x faster than Dijkstra")

========== AlgoKit ==========
Dijkstra     : 0.094059 s
Bellman-Ford : 0.105563 s

======= NetworkX =======
Dijkstra     : 1.024206 s
Bellman-Ford : 2.329434 s

======== Speedups ========
Dijkstra     : 10.89x
Bellman-Ford : 22.07x

==== AlgoKit Internal ====
Dijkstra is 1.12x faster than Bellman-Ford


## Floyd-Warshall

A small 4-vertex weighted DAG. Prints the full `distance(i, j)` matrix for every pair.

In [ ]:
import algokit

g = algokit.Graph.directed(4)

g.add_edges([
    (0,1,5),
    (0,3,10),
    (1,2,3),
    (2,3,1)
])

fw = g.floyd_warshall()

print(fw)

for i in range(4):
    print([fw.distance(i,j) for j in range(4)])

[0.0, 5.0, 8.0, 9.0]
[inf, 0.0, 3.0, 4.0]
[inf, inf, 0.0, 1.0]
[inf, inf, inf, 0.0]


Reconstructs the actual shortest path (not just its length) for three
specific `(from, to)` pairs using `fw.path(...)`.

In [ ]:
print(fw.path(0,3))
print(fw.path(0,2))
print(fw.path(1,3))

[0, 1, 2, 3]
[0, 1, 2]
[1, 2, 3]


### Reachability across all pairs

A sparser 5-vertex graph (`0→1→2` only) — prints `is_reachable(i, j)` for every ordered
pair, making the "who can reach whom" structure fully visible at once.

In [ ]:
g = algokit.Graph.directed(5)

g.add_edges([
    (0,1,2),
    (1,2,3)
])

fw = g.floyd_warshall()

for i in range(5):
    for j in range(5):
        print(
            i,
            "->",
            j,
            fw.is_reachable(i,j)
        )

0 -> 0 True
0 -> 1 True
0 -> 2 True
0 -> 3 False
0 -> 4 False
1 -> 0 False
1 -> 1 True
1 -> 2 True
1 -> 3 False
1 -> 4 False
2 -> 0 False
2 -> 1 False
2 -> 2 True
2 -> 3 False
2 -> 4 False
3 -> 0 False
3 -> 1 False
3 -> 2 False
3 -> 3 True
3 -> 4 False
4 -> 0 False
4 -> 1 False
4 -> 2 False
4 -> 3 False
4 -> 4 True


### Edge case — no edges at all

4 vertices, zero edges. Every vertex can trivially reach itself (`distance(i,i) == 0`)
but nothing else.

**Expect:** all off-diagonal distances are `inf`, all off-diagonal `is_reachable` is
`False`.

In [ ]:
g = algokit.Graph.directed(4)

fw = g.floyd_warshall()

for i in range(4):
    print([fw.distance(i,j) for j in range(4)])

[0.0, inf, inf, inf]
[inf, 0.0, inf, inf]
[inf, inf, 0.0, inf]
[inf, inf, inf, 0.0]


### Negative edge, no negative cycle

One negative edge (`1→2` weight `-2`) mixed into an otherwise positive-weight graph.
Floyd-Warshall handles this correctly (unlike Dijkstra) as long as there's no negative
*cycle*.

In [ ]:
g = algokit.Graph.directed(4)

g.add_edges([
    (0,1,4),
    (0,2,5),
    (1,2,-2),
    (2,3,3)
])

fw = g.floyd_warshall()

for i in range(4):
    print([fw.distance(i,j) for j in range(4)])

[0.0, 4.0, 2.0, 5.0]
[inf, 0.0, -2.0, 1.0]
[inf, inf, 0.0, 3.0]
[inf, inf, inf, 0.0]


### Negative cycle — this should raise

The same 2-cycle (`1→2→1`, total weight `-4`) used to test Bellman-Ford's cycle
detection.

**Expect:** a `RuntimeError`.

In [ ]:
g = algokit.Graph.directed(3)

g.add_edges([
    (0,1,1),
    (1,2,-2),
    (2,1,-2)
])

try:
    fw = g.floyd_warshall()

    print("Succeeded")

except Exception as e:
    print(type(e).__name__, e)

RuntimeError Floyd-Warshall detected a negative-weight cycle.


### Error handling — out-of-range vertex ids

Both `distance()` and `path()` should raise when given a vertex id that doesn't exist in
the graph.

In [ ]:
try:
    fw.distance(100,0)
except Exception as e:
    print(type(e).__name__, e)

try:
    fw.path(0,100)
except Exception as e:
    print(type(e).__name__, e)

IndexError vector::_M_range_check: __n (which is 100) >= this->size() (which is 4)
IndexError vector::_M_range_check: __n (which is 100) >= this->size() (which is 4)


### Randomized cross-validation vs `networkx`

Only 30 iterations here (versus 500–1000 for the single-source algorithms above) — Floyd-
Warshall's `O(V³)` cost per graph, times an all-pairs `networkx` comparison, makes a large
sample much more expensive to run.

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 30

for test in range(NUM_TESTS):

    n = random.randint(3,15)

    g = algokit.Graph.directed(n)

    G = nx.DiGraph()

    G.add_nodes_from(range(n))

    edges = set()

    while len(edges) < n*3:

        u = random.randrange(n)
        v = random.randrange(n)

        if u == v or (u,v) in edges:
            continue

        edges.add((u,v))

        w = random.randint(1,20)

        g.add_edge(u,v,w)
        G.add_edge(u,v,weight=w)

    fw = g.floyd_warshall()

    failed = False

    for s in range(n):

        nx_dist = nx.single_source_dijkstra_path_length(
            G,
            s,
            weight="weight"
        )

        for v in range(n):

            if v in nx_dist:

                if abs(
                    fw.distance(s,v)
                    -
                    nx_dist[v]
                ) > 1e-9:

                    failed = True
                    break

            else:

                if fw.is_reachable(s,v):

                    failed = True
                    break

        if failed:
            break

    if failed:

        print("❌ Failed")

        print("Test",test)

        break

else:

    print(
        f"✅ All {NUM_TESTS} randomized Floyd-Warshall tests passed!"
    )

KeyboardInterrupt: 

### Triple cross-check: Floyd-Warshall vs Dijkstra vs Bellman-Ford

300 random graphs, each checked with all three shortest-path algorithms at once: for
every source vertex, `dijkstra()` and `bellman_ford()`'s single-source results are
compared against the matching row of `floyd_warshall()`'s all-pairs result. All three
should agree exactly on every reachable pair.

In [ ]:
import random
import algokit

NUM_TESTS = 300

for test in range(NUM_TESTS):

    n = random.randint(5, 25)

    g = algokit.Graph.directed(n)

    edges = set()

    while len(edges) < n * 3:

        u = random.randrange(n)
        v = random.randrange(n)

        if u == v or (u, v) in edges:
            continue

        edges.add((u, v))

        w = random.randint(1, 20)

        g.add_edge(u, v, w)

    fw = g.floyd_warshall()

    for source in range(n):

        dijkstra = g.dijkstra(source)
        bellman = g.bellman_ford(source)

        for v in range(n):

            assert fw.is_reachable(source, v) == dijkstra.is_reachable(v)
            assert fw.is_reachable(source, v) == bellman.is_reachable(v)

            if fw.is_reachable(source, v):

                assert abs(
                    fw.distance(source, v)
                    - dijkstra.distance(v)
                ) < 1e-9

                assert abs(
                    fw.distance(source, v)
                    - bellman.distance(v)
                ) < 1e-9

print("✅ Floyd-Warshall, Dijkstra and Bellman-Ford all agree!")

✅ Floyd-Warshall, Dijkstra and Bellman-Ford all agree!


## Minimum spanning tree — Kruskal (`kruskal`)

A 4-vertex weighted undirected graph with 5 candidate edges — more edges than a spanning
tree needs, so Kruskal has to actually choose. Prints the resulting `MSTResult`.

In [ ]:
import algokit

g = algokit.Graph.undirected(4)

g.add_edges([
    (0,1,10),
    (0,2,6),
    (0,3,5),
    (1,3,15),
    (2,3,4)
])

mst = g.kruskal()

print(mst)
print("Total Weight:", mst.total_weight())
print("Edge Count  :", mst.edge_count())
print("Edges       :", mst.edges())

MSTResult(total_weight=19, edge_count=3)
Total Weight: 19.0
Edge Count  : 3
Edges       : [GraphEdge(from_vertex=2, to_vertex=3, weight=4), GraphEdge(from_vertex=0, to_vertex=3, weight=5), GraphEdge(from_vertex=0, to_vertex=1, weight=10)]


### Edge case — single-vertex graph

No edges needed to "span" one vertex.

**Expect:** `total_weight() == 0.0`, `edge_count() == 0`.

In [ ]:
g = algokit.Graph.undirected(1)

mst = g.kruskal()

print(mst.total_weight())
print(mst.edge_count())
print(mst.edges())

0.0
0
[]


A graph that's already a minimal chain (`0-1-2-3-4`, no redundant edges) — the
MST should be the entire graph, unchanged.

In [ ]:
g = algokit.Graph.undirected(5)

g.add_edges([
    (0,1,2),
    (1,2,3),
    (2,3,4),
    (3,4,5)
])

mst = g.kruskal()

print(mst.total_weight())
print(mst.edge_count())
print(mst.edges())

14.0
4
[GraphEdge(from_vertex=0, to_vertex=1, weight=2), GraphEdge(from_vertex=1, to_vertex=2, weight=3), GraphEdge(from_vertex=2, to_vertex=3, weight=4), GraphEdge(from_vertex=3, to_vertex=4, weight=5)]


### Redundant equal-weight edges

A 4-cycle plus one diagonal, all with weight `1`. Multiple different spanning trees have
the same minimum total weight here — the test only checks `total_weight()` and
`edge_count()`, not which *specific* edges were chosen, since any valid MST is an
acceptable answer.

In [ ]:
g = algokit.Graph.undirected(4)

g.add_edges([
    (0,1,1),
    (1,2,1),
    (2,3,1),
    (3,0,1),
    (0,2,1)
])

mst = g.kruskal()

print(mst.total_weight())
print(mst.edge_count())
print(mst.edges())

3.0
3
[GraphEdge(from_vertex=0, to_vertex=1, weight=1), GraphEdge(from_vertex=1, to_vertex=2, weight=1), GraphEdge(from_vertex=2, to_vertex=3, weight=1)]


### Disconnected graph — no exception, just a partial forest

Two separate components (`0-1` and `3-4`); vertex `2` is isolated. `kruskal()` doesn't
require the input to be connected — it simply never finds enough safe edges to span
everything, and returns whatever spanning *forest* it could build.

**Expect:** no exception; `mst.edge_count() == 2` (one edge per connected component that
has any edges at all), not 4 (which would be needed to span all 5 vertices as a single
tree).

In [ ]:
g = algokit.Graph.undirected(5)

g.add_edges([
    (0,1,2),
    (3,4,5)
])

try:
    mst = g.kruskal()
    print(mst)
except Exception as e:
    print(type(e).__name__, e)

MSTResult(total_weight=7, edge_count=2)


### Error handling — wrong graph type

`kruskal()` is only defined for undirected graphs.

**Expect:** a `RuntimeError` when called on a directed graph.

In [ ]:
g = algokit.Graph.directed(4)

try:
    g.kruskal()
except Exception as e:
    print(type(e).__name__, e)

RuntimeError Kruskal's algorithm is only supported for undirected graphs.


## Minimum spanning tree — Prim (`prim`)

The exact same graph used to introduce Kruskal above — since both algorithms compute a
*minimum* spanning tree, they must produce the same `total_weight()` (though not
necessarily the identical edge set, when ties exist).

In [ ]:
g = algokit.Graph.undirected(4)

g.add_edges([
    (0,1,10),
    (0,2,6),
    (0,3,5),
    (1,3,15),
    (2,3,4)
])

mst = g.prim()

print(mst)
print(mst.total_weight())
print(mst.edge_count())
print(mst.edges())

MSTResult(total_weight=19, edge_count=3)
19.0
3
[GraphEdge(from_vertex=0, to_vertex=3, weight=5), GraphEdge(from_vertex=3, to_vertex=2, weight=4), GraphEdge(from_vertex=0, to_vertex=1, weight=10)]


Single-vertex graph — mirrors the Kruskal edge case.

In [ ]:
g = algokit.Graph.undirected(1)

mst = g.prim()

print(mst.total_weight())
print(mst.edge_count())
print(mst.edges())

0.0
0
[]


Already-minimal chain graph — mirrors the Kruskal test.

In [ ]:
g = algokit.Graph.undirected(5)

g.add_edges([
    (0,1,2),
    (1,2,3),
    (2,3,4),
    (3,4,5)
])

mst = g.prim()

print(mst.total_weight())
print(mst.edge_count())
print(mst.edges())

14.0
4
[GraphEdge(from_vertex=0, to_vertex=1, weight=2), GraphEdge(from_vertex=1, to_vertex=2, weight=3), GraphEdge(from_vertex=2, to_vertex=3, weight=4), GraphEdge(from_vertex=3, to_vertex=4, weight=5)]


Redundant equal-weight-edges graph — mirrors the Kruskal test.

In [ ]:
g = algokit.Graph.undirected(4)

g.add_edges([
    (0,1,1),
    (1,2,1),
    (2,3,1),
    (3,0,1),
    (0,2,1)
])

mst = g.prim()

print(mst.total_weight())
print(mst.edge_count())
print(mst.edges())

3.0
3
[GraphEdge(from_vertex=0, to_vertex=1, weight=1), GraphEdge(from_vertex=0, to_vertex=3, weight=1), GraphEdge(from_vertex=0, to_vertex=2, weight=1)]


### Disconnected graph — Prim also returns a partial forest

Same disconnected graph as the Kruskal test above. Prim's implementation explicitly
restarts from every unvisited vertex, so — like Kruskal — it does **not** raise on a
disconnected graph; it builds a minimum spanning *forest* instead.

**Expect:** no exception; same `total_weight()` and `edge_count()` as Kruskal produced on
this graph.

In [ ]:
g = algokit.Graph.undirected(5)

g.add_edges([
    (0,1,2),
    (3,4,5)
])

try:
    mst = g.prim()
    print(mst)
    print(mst.total_weight())
    print(mst.edge_count())
    print(mst.edges())
except Exception as e:
    print(type(e).__name__, e)

MSTResult(total_weight=7, edge_count=2)
7.0
2
[GraphEdge(from_vertex=0, to_vertex=1, weight=2), GraphEdge(from_vertex=3, to_vertex=4, weight=5)]


### Error handling — wrong graph type

`prim()` is only defined for undirected graphs, same restriction as Kruskal.

**Expect:** a `RuntimeError` when called on a directed graph.

In [ ]:
g = algokit.Graph.directed(4)

try:
    g.prim()
except Exception as e:
    print(type(e).__name__, e)

RuntimeError Prim's algorithm is only supported for undirected graphs.


### Self-loop doesn't affect the MST

Adds a self-loop on vertex `0` with a very high weight (`100`) alongside two normal edges.
A self-loop can never be part of any spanning tree (it doesn't connect two different
vertices), so it should be silently ignored rather than corrupting the total weight.

In [ ]:
g = algokit.Graph.undirected(3)

g.add_edge(0,0,100)
g.add_edge(0,1,2)
g.add_edge(1,2,3)

mst = g.prim()

print(mst.total_weight())
print(mst.edges())

5.0
[GraphEdge(from_vertex=0, to_vertex=1, weight=2), GraphEdge(from_vertex=1, to_vertex=2, weight=3)]


### Parallel edges (multigraph) — Prim should use the cheaper one

Two edges between the same pair, `0-1` with weight `10` and weight `2`. A correct MST
implementation should be able to use the cheaper of the two when it's relevant, rather
than only ever considering the first one added.

In [ ]:
g = algokit.Graph.undirected(3)

g.add_edge(0,1,10)
g.add_edge(0,1,2)
g.add_edge(1,2,3)

mst = g.prim()

print(mst.total_weight())
print(mst.edges())

5.0
[GraphEdge(from_vertex=0, to_vertex=1, weight=2), GraphEdge(from_vertex=1, to_vertex=2, weight=3)]


### Benchmark: Kruskal vs Prim

A single medium-sized graph (1000 vertices, 5000 edges, built starting from a guaranteed-
connected chain so both algorithms always succeed), timing one call to each algorithm and
printing the ratio between them.

In [ ]:
import random
import time
import algokit

n = 1000
m = 5000

g = algokit.Graph.undirected(n)

edges = set()

# Ensure connected
for i in range(1, n):
    w = random.randint(1, 100)
    g.add_edge(i - 1, i, w)
    edges.add((i - 1, i))

while len(edges) < m:
    u = random.randrange(n)
    v = random.randrange(n)

    if u == v:
        continue

    e = tuple(sorted((u, v)))

    if e in edges:
        continue

    edges.add(e)

    g.add_edge(u, v, random.randint(1, 100))

start = time.perf_counter()
g.kruskal()
kruskal_time = time.perf_counter() - start

start = time.perf_counter()
g.prim()
prim_time = time.perf_counter() - start

print(f"Kruskal : {kruskal_time:.6f} s")
print(f"Prim    : {prim_time:.6f} s")
print(f"Ratio   : {prim_time/kruskal_time:.2f}x")

Kruskal : 0.000961 s
Prim    : 0.001560 s
Ratio   : 1.62x


## Aside

Not part of this notebook's main topic — a leftover strongly-connected-components check
(identical to the one in `DSU_SSCTesting.ipynb`), likely kept here for a quick
side-by-side reference while developing the other sections above.

In [ ]:
import algokit

g = algokit.Graph.directed(4)

g.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,0)
])

scc = g.strongly_connected_components()

print(scc)
print("Count:", scc.component_count())
print("IDs:", scc.component_ids())
print("Components:", scc.components())

StronglyConnectedComponentsResult(component_count=1)
Count: 1


AttributeError: 'algokit.StronglyConnectedComponentsResult' object has no attribute 'component_ids'

Prints the built-in `help()` text for `StronglyConnectedComponentsResult`.

In [ ]:
help(algokit.StronglyConnectedComponentsResult)

Help on class StronglyConnectedComponentsResult in module algokit:

class StronglyConnectedComponentsResult(pybind11_builtins.pybind11_object)
 |  Result of strongly connected components (SCC) computation.
 |
 |  Attributes:
 |      component_count (int): Number of SCCs.
 |      component_id (list[int]): Component ID for each vertex.
 |      components (list[list[int]]): List of vertices in each SCC.
 |
 |  Method resolution order:
 |      StronglyConnectedComponentsResult
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __repr__(...)
 |      __repr__(self: algokit.StronglyConnectedComponentsResult) -> str
 |
 |      Returns a concise representation of the SCC result.
 |
 |  component_count(...)
 |      component_count(self: algokit.StronglyConnectedComponentsResult) -> int
 |
 |      Returns the number of strongly connect